# NUS ST3232 — Design and Analysis of Experiments
## Kaggle case study: Concrete Compressive Strength

This notebook is a **tutorial-style implementation** of the main ideas in NUS ST3232 using the
**Concrete Compressive Strength** dataset available on Kaggle.

**Kaggle dataset used**

- Dataset: `vivekgediya/concrete-data`
- File: `Concrete_Data.csv`
- 1,030 observations
- 8 quantitative inputs: cement, blast furnace slag, fly ash, water, superplasticizer,
  coarse aggregate, fine aggregate, and age
- Response: concrete compressive strength (MPa)
- Original source: Prof. I-Cheng Yeh / UCI Machine Learning Repository

Kaggle page: https://www.kaggle.com/datasets/vivekgediya/concrete-data

---

## Important statistical caveat

The source data contains laboratory measurements, but the public dataset does **not document a
randomized factorial assignment mechanism** for the observations. Therefore:

- ANOVA, regression, interactions, blocking-style adjustments and ANCOVA on the observed data are
  useful for learning the **analysis machinery**.
- They do **not automatically justify causal statements** such as "increasing cement causes exactly X MPa".
- For full/fractional factorial and causal-inference demonstrations, we construct **designed experiments
  anchored to realistic factor ranges from the Kaggle data**.

That distinction is one of the most important lessons in Design of Experiments:

> A sophisticated model cannot repair an experiment whose treatment assignment does not identify the
> causal effect of interest.

---

## Learning objectives

By the end you will have implemented:

1. Experimental units, response, factors, levels and treatments
2. Randomization, replication and blocking
3. One-way ANOVA
4. ANOVA decomposition and effect size
5. Residual diagnostics
6. Multiple comparisons using Tukey HSD
7. Randomized-block style analysis
8. Two-factor models and interactions
9. ANCOVA
10. \(2^k\) full factorial designs
11. Factorial effect estimation with \(-1/+1\) coding
12. Blocking/confounding intuition
13. Fractional factorial designs, generators and aliasing
14. Power / sample-size thinking
15. Randomized causal inference vs confounding
16. A practical DOE workflow for engineering/data-science problems

## 0. Environment setup

The notebook uses:

- **pandas / NumPy** — data manipulation
- **SciPy** — statistical tests
- **statsmodels** — OLS, ANOVA, Tukey HSD and power analysis
- **scikit-learn** — a response-surface surrogate used only for the designed-experiment demonstrations
- **Bokeh** — interactive visualisation
- **kagglehub** — downloading the public Kaggle dataset

If you run this on Kaggle and the dataset is already attached, the loader also searches `/kaggle/input`.

In [1]:
# Run once if your environment is missing packages.
# In a managed environment you may comment this cell out.

%pip install -q kagglehub bokeh statsmodels scipy scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from itertools import product
from typing import Iterable, Sequence
import warnings

import numpy as np
import pandas as pd

from scipy import stats
from scipy.stats import levene

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.power import FTestAnovaPower

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.models import (
    ColumnDataSource,
    ColorBar,
    HoverTool,
    LinearColorMapper,
    Span,
)
from bokeh.palettes import Viridis256
from bokeh.plotting import figure

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

output_notebook()

Loading BokehJS ...

## 1. Load the Kaggle dataset

The loader is deliberately robust:

1. Try `kagglehub.dataset_download(...)`
2. Search the downloaded directory for `Concrete_Data.csv`
3. If running inside a Kaggle notebook, search `/kaggle/input`
4. Search the current working directory

For a public dataset, `kagglehub` usually handles the download directly.

In [3]:
@dataclass(frozen=True)
class KaggleConcreteLoader:
    dataset_handle: str = "vivekgediya/concrete-data"
    filename: str = "Concrete_Data.csv"

    def locate(self) -> Path:
        candidates: list[Path] = []

        # 1) KaggleHub
        try:
            import kagglehub
            dataset_dir = Path(kagglehub.dataset_download(self.dataset_handle))
            candidates.extend(dataset_dir.rglob(self.filename))
            candidates.extend(dataset_dir.rglob("*.csv"))
        except Exception as exc:
            print(f"KaggleHub download skipped/failed: {type(exc).__name__}: {exc}")

        # 2) Kaggle notebook filesystem
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            candidates.extend(kaggle_root.rglob(self.filename))
            candidates.extend(kaggle_root.rglob("*.csv"))

        # 3) Current directory
        candidates.extend(Path(".").rglob(self.filename))

        # Prefer exact filename.
        exact = [p for p in candidates if p.name.lower() == self.filename.lower()]
        if exact:
            return exact[0]

        if candidates:
            return candidates[0]

        raise FileNotFoundError(
            "Concrete_Data.csv was not found. Download the Kaggle dataset "
            "'vivekgediya/concrete-data' and place Concrete_Data.csv beside this notebook."
        )

    def load(self) -> pd.DataFrame:
        path = self.locate()
        print(f"Loading: {path}")
        return pd.read_csv(path)


raw_df = KaggleConcreteLoader().load()
print("Raw shape:", raw_df.shape)
display(raw_df.head())

100%|█████████████████████████████████████████████████████████████████████████████| 11.2k/11.2k [00:00<00:00, 8.71MB/s]

Extracting files...


Loading: /home/anirban/.cache/kagglehub/datasets/vivekgediya/concrete-data/versions/1/Concrete_Data.csv
Raw shape: (1030, 9)


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0000,0.0000,0.0000,162.0000,2.5000,"1,040.0000",676.0000,28,79.9900
1,540.0000,0.0000,0.0000,162.0000,2.5000,"1,055.0000",676.0000,28,61.8900
2,332.5000,142.5000,0.0000,228.0000,0.0000,932.0000,594.0000,270,40.2700
3,332.5000,142.5000,0.0000,228.0000,0.0000,932.0000,594.0000,365,41.0500
4,198.6000,132.4000,0.0000,192.0000,0.0000,978.4000,825.5000,360,44.3000


### 1.1 Normalize column names

Kaggle mirrors of this dataset sometimes preserve long UCI-style headings.
Rather than assuming exact spelling, we map columns using semantic keywords.

In [57]:
import pandas as pd

def normalize_concrete_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Canonical mapping dictionary
    mapping = {
        "compressive strength": "strength",
        "cement": "cement",
        "blast furnace slag": "slag",
        "slag": "slag",
        "fly ash": "fly_ash",
        "superplasticizer": "superplasticizer",
        "coarse aggregate": "coarse_aggregate",
        "fine aggregate": "fine_aggregate",
        "water": "water",
        "age": "age",
    }

    # Normalize columns directly with dict lookup + fallback
    out.columns = [
        next((val for key, val in mapping.items() if key in str(c).lower()), 
             str(c).strip().lower()
                 .replace(" ", "_")
                 .replace("(", "")
                 .replace(")", "")
                 .replace("/", "_")
                 .replace("^", "")
        )
        for c in out.columns
    ]

    # Expected schema derived from mapping values
    expected = set(mapping.values())

    missing = expected.difference(out.columns)
    if missing:
        raise ValueError(
            f"Could not identify expected columns: {sorted(missing)}\n"
            f"Detected columns: {list(out.columns)}"
        )

    return out[list(sorted(expected - {"strength"})) + ["strength"]]



df = normalize_concrete_columns(raw_df)
print(df.columns.tolist())
display(df.head())

['age', 'cement', 'coarse_aggregate', 'fine_aggregate', 'fly_ash', 'slag', 'superplasticizer', 'water', 'strength']


,age,cement,coarse_aggregate,fine_aggregate,fly_ash,slag,superplasticizer,water,strength
0,28,540.0000,"1,040.0000",676.0000,0.0000,0.0000,2.5000,162.0000,79.9900
1,28,540.0000,"1,055.0000",676.0000,0.0000,0.0000,2.5000,162.0000,61.8900
2,270,332.5000,932.0000,594.0000,0.0000,142.5000,0.0000,228.0000,40.2700
3,365,332.5000,932.0000,594.0000,0.0000,142.5000,0.0000,228.0000,41.0500
4,360,198.6000,978.4000,825.5000,0.0000,132.4000,0.0000,192.0000,44.3000


## 2. Understand the experimental vocabulary in this case study

For DOE language, we can interpret the columns as follows:

- **Response**: `strength` (MPa)
- **Candidate controllable factors**: cement, water, slag, fly ash, superplasticizer, aggregates
- **Time/covariate**: age in days
- **Experimental unit**: an individual concrete specimen / mixture observation
- **Treatment**: a particular combination of factor settings

However, we do **not** know that these factor settings were randomly assigned according to a DOE plan.
So for the raw dataset, we will say **factor-like variables** rather than claiming randomized treatments.

In [5]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique(),
    "min": df.min(numeric_only=True),
    "median": df.median(numeric_only=True),
    "max": df.max(numeric_only=True),
})
display(summary)

,dtype,missing,n_unique,min,median,max
age,int64,0,14,1.0000,28.0000,365.0000
cement,float64,0,278,102.0000,272.9000,540.0000
coarse_aggregate,float64,0,284,801.0000,968.0000,"1,145.0000"
fine_aggregate,float64,0,302,594.0000,779.5000,992.6000
fly_ash,float64,0,156,0.0000,0.0000,200.1000
slag,float64,0,185,0.0000,22.0000,359.4000
superplasticizer,float64,0,111,0.0000,6.4000,32.2000
water,float64,0,195,121.8000,185.0000,247.0000
strength,float64,0,845,2.3300,34.4450,82.6000


## 3. Exploratory analysis

Before designing or analysing an experiment, understand:

- range of feasible factor settings,
- skewness,
- sparsity,
- possible nonlinearities,
- obvious correlations,
- whether all treatment regions have data support.

This is not a substitute for experimental design; it helps define sensible factor levels.

In [6]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
age,"1,030.0000",45.6621,63.1699,1.0000,7.0000,28.0000,56.0000,365.0000
cement,"1,030.0000",281.1679,104.5064,102.0000,192.3750,272.9000,350.0000,540.0000
coarse_aggregate,"1,030.0000",972.9189,77.7540,801.0000,932.0000,968.0000,"1,029.4000","1,145.0000"
fine_aggregate,"1,030.0000",773.5805,80.1760,594.0000,730.9500,779.5000,824.0000,992.6000
fly_ash,"1,030.0000",54.1883,63.9970,0.0000,0.0000,0.0000,118.3000,200.1000
slag,"1,030.0000",73.8958,86.2793,0.0000,0.0000,22.0000,142.9500,359.4000
superplasticizer,"1,030.0000",6.2047,5.9738,0.0000,0.0000,6.4000,10.2000,32.2000
water,"1,030.0000",181.5673,21.3542,121.8000,164.9000,185.0000,192.0000,247.0000
strength,"1,030.0000",35.8180,16.7057,2.3300,23.7100,34.4450,46.1350,82.6000


In [7]:
def bokeh_histogram(series: pd.Series, title: str, x_label: str, bins: int = 30):
    values = series.dropna().to_numpy()
    hist, edges = np.histogram(values, bins=bins)

    p = figure(
        width=760,
        height=360,
        title=title,
        x_axis_label=x_label,
        y_axis_label="Count",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p.quad(
        top=hist,
        bottom=0,
        left=edges[:-1],
        right=edges[1:],
        line_color="white",
        fill_alpha=0.75,
    )
    return p

show(bokeh_histogram(df["strength"], "Distribution of concrete compressive strength", "Strength (MPa)"))

In [8]:
def bokeh_scatter(df_: pd.DataFrame, x: str, y: str, title: str):
    src = ColumnDataSource(df_)
    p = figure(
        width=760,
        height=400,
        title=title,
        x_axis_label=x,
        y_axis_label=y,
        tools="pan,wheel_zoom,box_zoom,reset,save,hover",
        tooltips=[(x, f"@{x}{{0.00}}"), (y, f"@{y}{{0.00}}"), ("age", "@age{0}")],
    )
    p.scatter(x=x, y=y, source=src, size=6, alpha=0.5)
    return p

p1 = bokeh_scatter(df, "cement", "strength", "Cement vs compressive strength")
p2 = bokeh_scatter(df, "water", "strength", "Water vs compressive strength")
show(gridplot([[p1], [p2]]))

### 3.1 Correlation structure

Correlation is **not causation**, but it is useful for identifying:

- strongly related inputs,
- possible confounding,
- collinearity,
- candidate interactions,
- factor regions worth studying in a controlled experiment.

In [9]:
corr = df.corr(numeric_only=True).round(3)
display(corr)

,age,cement,coarse_aggregate,fine_aggregate,fly_ash,slag,superplasticizer,water,strength
age,1.0000,0.0820,-0.0030,-0.1560,-0.1540,-0.0440,-0.1930,0.2780,0.3290
cement,0.0820,1.0000,-0.1090,-0.2230,-0.3970,-0.2750,0.0920,-0.0820,0.4980
coarse_aggregate,-0.0030,-0.1090,1.0000,-0.1780,-0.0100,-0.2840,-0.2660,-0.1820,-0.1650
fine_aggregate,-0.1560,-0.2230,-0.1780,1.0000,0.0790,-0.2820,0.2230,-0.4510,-0.1670
fly_ash,-0.1540,-0.3970,-0.0100,0.0790,1.0000,-0.3240,0.3780,-0.2570,-0.1060
slag,-0.0440,-0.2750,-0.2840,-0.2820,-0.3240,1.0000,0.0430,0.1070,0.1350
superplasticizer,-0.1930,0.0920,-0.2660,0.2230,0.3780,0.0430,1.0000,-0.6580,0.3660
water,0.2780,-0.0820,-0.1820,-0.4510,-0.2570,0.1070,-0.6580,1.0000,-0.2900
strength,0.3290,0.4980,-0.1650,-0.1670,-0.1060,0.1350,0.3660,-0.2900,1.0000


In [10]:
def bokeh_corr_heatmap(corr_df: pd.DataFrame):
    names = list(corr_df.columns)
    records = [
        {"x": x, "y": y, "value": corr_df.loc[y, x]}
        for y in names
        for x in names
    ]
    hdf = pd.DataFrame(records)

    mapper = LinearColorMapper(palette=Viridis256, low=-1, high=1)

    p = figure(
        width=760,
        height=650,
        title="Correlation heatmap",
        x_range=names,
        y_range=list(reversed(names)),
        tools="hover,save,reset",
        tooltips=[("x", "@x"), ("y", "@y"), ("correlation", "@value{0.000}")],
    )
    p.rect(
        x="x",
        y="y",
        width=1,
        height=1,
        source=ColumnDataSource(hdf),
        fill_color={"field": "value", "transform": mapper},
        line_color=None,
    )
    p.add_layout(ColorBar(color_mapper=mapper), "right")
    p.xaxis.major_label_orientation = 1.0
    return p

show(bokeh_corr_heatmap(corr))

# Part I — Single-factor experiments and ANOVA

## 4. Create pedagogical factor levels for cement

The raw `cement` variable is continuous. To illustrate one-way ANOVA, we create three **analysis strata**:

\[
\text{Low},\quad \text{Medium},\quad \text{High}
\]

using empirical tertiles.

This is **not equivalent to a randomized three-treatment experiment**. It is a convenient way to learn
the ANOVA calculations on real measurements.

In [11]:
anova_df = df.copy()
anova_df["cement_level"] = pd.qcut(
    anova_df["cement"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop",
)

group_summary = (
    anova_df.groupby("cement_level", observed=True)["strength"]
    .agg(["count", "mean", "std", "median"])
)
display(group_summary)

,count,mean,std,median
cement_level,,,,
Low,344,27.6813,13.0533,27.1000
Medium,342,34.2594,14.7263,33.4100
High,344,45.5041,16.9807,42.8150


In [55]:
def bokeh_box_data(df_: pd.DataFrame, group: str, y: str) -> pd.DataFrame:
    rows = []
    for name, g in df_.groupby(group, observed=True):
        q1, q2, q3 = np.percentile(g[y], [25, 50, 75])
        iqr = q3 - q1
        low = max(g[y].min(), q1 - 1.5 * iqr)
        high = min(g[y].max(), q3 + 1.5 * iqr)
        rows.append({
            group: str(name), "q1": q1, "median": q2, "q3": q3,
            "low": low, "high": high,
        })
    return pd.DataFrame(rows)


def show_bokeh_boxplot(df_: pd.DataFrame, group: str, y: str, title: str):
    box = bokeh_box_data(df_, group, y)
    cats = box[group].tolist()
    src = ColumnDataSource(box)

    p = figure(
        x_range=cats,
        width=760,
        height=400,
        title=title,
        x_axis_label=group,
        y_axis_label=y,
        tools="pan,wheel_zoom,reset,save",
    )

    p.segment(x0=group, y0="high", x1=group, y1="q3", source=src)
    p.segment(x0=group, y0="low", x1=group, y1="q1", source=src)
    p.vbar(x=group, width=0.55, top="q3", bottom="q2", source=src, alpha=0.6)
    p.vbar(x=group, width=0.55, top="q2", bottom="q1", source=src, alpha=0.4)
    p.segment(x0=group, y0="median", x1=group, y1="median", source=src, line_width=3)

    show(p)


show_bokeh_boxplot(
    anova_df,
    "cement_level",
    "strength",
    "Observed strength by cement analysis stratum",
)

ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : top='q2' [no close matches] {renderer: GlyphRenderer(id='p1831', ...)}
ERROR:bokeh.core.validation.check:E-1001 (BAD_COLUMN_NAME): Glyph refers to nonexistent column name. This could either be due to a misspelling or typo, or due to an expected column being missing. : bottom='q2' [no close matches] {renderer: GlyphRenderer(id='p1822', ...)}


## 5. One-way ANOVA model

We fit

\[
Y_{ij} = \mu + \tau_i + \varepsilon_{ij}
\]

and test

\[
H_0:\mu_{\text{Low}}=\mu_{\text{Medium}}=\mu_{\text{High}}.
\]

The ANOVA \(F\)-statistic compares between-group variation with within-group variation:

\[
F=\frac{MS_{\text{factor}}}{MS_E}.
\]

In [13]:
one_way = smf.ols("strength ~ C(cement_level)", data=anova_df).fit()
one_way_table = anova_lm(one_way, typ=2)
display(one_way_table)

,sum_sq,df,F,PR(>F)
C(cement_level),"55,880.3050",2.0000,124.0604,0.0000
Residual,"231,294.8821","1,027.0000",NaN,NaN


### 5.1 Manual ANOVA decomposition

It is useful to compute the sums of squares once by hand:

\[
SS_T = SS_A + SS_E
\]

so that ANOVA is not just a black-box function call.

In [14]:
grand_mean = anova_df["strength"].mean()

ss_total = ((anova_df["strength"] - grand_mean) ** 2).sum()

ss_factor = sum(
    len(g) * (g["strength"].mean() - grand_mean) ** 2
    for _, g in anova_df.groupby("cement_level", observed=True)
)

ss_error = sum(
    ((g["strength"] - g["strength"].mean()) ** 2).sum()
    for _, g in anova_df.groupby("cement_level", observed=True)
)

print(f"SS Total   = {ss_total:,.4f}")
print(f"SS Factor  = {ss_factor:,.4f}")
print(f"SS Error   = {ss_error:,.4f}")
print(f"Check      = {ss_factor + ss_error:,.4f}")
print(f"Difference = {ss_total - (ss_factor + ss_error):.8f}")

SS Total   = 287,175.1871
SS Factor  = 55,880.3050
SS Error   = 231,294.8821
Check      = 287,175.1871
Difference = 0.00000000


### 5.2 Effect size

A tiny p-value can occur with a large dataset even when the practical effect is modest.

For one-way ANOVA, a common effect size is

\[
\eta^2=\frac{SS_{\text{factor}}}{SS_T}.
\]

It measures the proportion of total observed variation associated with the factor grouping.

In [15]:
eta_squared = ss_factor / ss_total
print(f"eta^2 = {eta_squared:.4f}")

eta^2 = 0.1946


## 6. ANOVA assumptions and model diagnostics

Classical ANOVA assumes approximately:

1. independent errors,
2. constant error variance,
3. normally distributed errors.

**Independence is primarily a design property.** A residual plot cannot prove that observations were
independently randomized.

We can still diagnose variance and distributional behaviour.

In [16]:
diagnostics = pd.DataFrame({
    "fitted": one_way.fittedvalues,
    "residual": one_way.resid,
})

src = ColumnDataSource(diagnostics)

p = figure(
    width=760,
    height=400,
    title="Residuals vs fitted values",
    x_axis_label="Fitted strength",
    y_axis_label="Residual",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover",
    tooltips=[("fitted", "@fitted{0.00}"), ("residual", "@residual{0.00}")],
)
p.scatter("fitted", "residual", source=src, size=6, alpha=0.5)
p.add_layout(Span(location=0, dimension="width", line_dash="dashed"))
show(p)

In [17]:
# Q-Q plot using theoretical normal quantiles, rendered with Bokeh.
resid = np.sort(one_way.resid.to_numpy())
n = len(resid)
probs = (np.arange(1, n + 1) - 0.5) / n
theoretical = stats.norm.ppf(probs)

qq = pd.DataFrame({"theoretical": theoretical, "residual": resid})

slope, intercept, *_ = stats.linregress(theoretical, resid)
line_x = np.array([theoretical.min(), theoretical.max()])
line_y = intercept + slope * line_x

p = figure(
    width=760,
    height=400,
    title="Normal Q-Q plot of one-way ANOVA residuals",
    x_axis_label="Theoretical normal quantile",
    y_axis_label="Ordered residual",
    tools="pan,wheel_zoom,reset,save",
)
p.scatter("theoretical", "residual", source=ColumnDataSource(qq), size=5, alpha=0.5)
p.line(line_x, line_y, line_width=2)
show(p)

In [18]:
groups = [
    g["strength"].to_numpy()
    for _, g in anova_df.groupby("cement_level", observed=True)
]

lev_stat, lev_p = levene(*groups, center="median")
print(f"Levene/Brown-Forsythe style test: statistic={lev_stat:.4f}, p={lev_p:.6g}")

# Shapiro-Wilk is hypersensitive for large n, so treat it as a supplement, not the main diagnostic.
shap_stat, shap_p = stats.shapiro(one_way.resid.sample(min(5000, len(one_way.resid)), random_state=RANDOM_STATE))
print(f"Shapiro-Wilk: statistic={shap_stat:.4f}, p={shap_p:.6g}")

Levene/Brown-Forsythe style test: statistic=12.2378, p=5.59133e-06
Shapiro-Wilk: statistic=0.9915, p=1.20648e-05


### Interpretation note

If residual assumptions are poor, possible responses include:

- transform the response,
- use heteroskedasticity-robust standard errors,
- use a more suitable mean/variance model,
- redesign the experiment,
- model an omitted blocking/covariate structure,
- use nonparametric or permutation methods where appropriate.

DOE is not just about selecting a test after collecting data; **design choices control how much
unexplained noise remains**.

## 7. Multiple comparisons — Tukey HSD

ANOVA answers:

> Is there evidence that at least one group mean differs?

It does not directly answer:

> Which pairs differ?

Tukey's Honest Significant Difference procedure performs all pairwise comparisons while controlling
the family-wise error rate under its standard assumptions.

In [19]:
tukey = pairwise_tukeyhsd(
    endog=anova_df["strength"],
    groups=anova_df["cement_level"],
    alpha=0.05,
)
print(tukey)

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj  lower    upper   reject
-----------------------------------------------------
  High    Low -17.8229   0.0 -20.5086 -15.1371   True
  High Medium -11.2447   0.0 -13.9344   -8.555   True
   Low Medium   6.5782   0.0   3.8885   9.2679   True
-----------------------------------------------------


In [20]:
tukey_df = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0],
)
display(tukey_df)

,group1,group2,meandiff,p-adj,lower,upper,reject
0,High,Low,-17.8229,0.0000,-20.5086,-15.1371,True
1,High,Medium,-11.2447,0.0000,-13.9344,-8.5550,True
2,Low,Medium,6.5782,0.0000,3.8885,9.2679,True


# Part II — Blocking and ANCOVA

## 8. Blocking intuition using curing age

Age is a major source of variation in concrete strength. In a **prospective randomized experiment**,
we might deliberately ensure each cement treatment is represented within the same curing-age blocks.

On the existing dataset, we can demonstrate the algebra by treating common age values as blocks.

A randomized complete block model has the structure

\[
Y_{ij}=\mu+\tau_i+\beta_j+\varepsilon_{ij}.
\]

Here:

- \(\tau_i\): factor/treatment effect
- \(\beta_j\): block effect

In [21]:
# Keep the most common exact curing ages so that the blocking demonstration has reasonable support.
common_ages = df["age"].value_counts().head(5).index.tolist()
block_df = df[df["age"].isin(common_ages)].copy()

block_df["cement_level"] = pd.qcut(
    block_df["cement"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop",
)
block_df["age_block"] = block_df["age"].astype(str) + " days"

cell_counts = pd.crosstab(block_df["age_block"], block_df["cement_level"])
display(cell_counts)

cement_level,Low,Medium,High
age_block,,,
14 days,24,30,8
28 days,162,142,121
3 days,42,42,50
56 days,25,30,36
7 days,27,35,64


In [22]:
unblocked_model = smf.ols("strength ~ C(cement_level)", data=block_df).fit()
blocked_model = smf.ols("strength ~ C(cement_level) + C(age_block)", data=block_df).fit()

print("UNBLOCKED")
display(anova_lm(unblocked_model, typ=2))

print("\nBLOCKED / AGE-ADJUSTED")
display(anova_lm(blocked_model, typ=2))

print(f"Residual variance, unblocked: {unblocked_model.mse_resid:.4f}")
print(f"Residual variance, blocked:   {blocked_model.mse_resid:.4f}")
print(
    "Relative residual-variance reduction: "
    f"{1 - blocked_model.mse_resid / unblocked_model.mse_resid:.2%}"
)

UNBLOCKED


,sum_sq,df,F,PR(>F)
C(cement_level),"54,240.7143",2.0000,131.7367,0.0000
Residual,"171,899.6334",835.0000,NaN,NaN



BLOCKED / AGE-ADJUSTED


,sum_sq,df,F,PR(>F)
C(cement_level),"61,931.4299",2.0000,278.7035,0.0000
C(age_block),"79,570.3081",4.0000,179.0410,0.0000
Residual,"92,329.3253",831.0000,NaN,NaN


Residual variance, unblocked: 205.8678
Residual variance, blocked:   111.1063
Relative residual-variance reduction: 46.03%


### What to look for

If age explains substantial variation, including it as a block-like term can reduce residual variance.

In a true designed experiment, blocking would be planned **before** data collection and treatments would
usually be randomized **within each block**.

## 9. ANCOVA — combine categorical treatments with continuous covariates

Blocking is natural when nuisance variables are categorical. If a nuisance predictor is continuous,
ANCOVA can preserve more information.

A simple ANCOVA model is

\[
Y_{ij} = \mu+\tau_i+\beta(X_{ij}-\bar X)+\varepsilon_{ij}.
\]

Here we use:

- factor-like variable: cement level
- continuous covariate: age

In [23]:
ancova_df = anova_df.copy()
ancova_df["age_centered"] = ancova_df["age"] - ancova_df["age"].mean()

ancova_model = smf.ols(
    "strength ~ C(cement_level) + age_centered",
    data=ancova_df,
).fit()

display(anova_lm(ancova_model, typ=2))
print(ancova_model.summary())

,sum_sq,df,F,PR(>F)
C(cement_level),"49,173.5153",2.0000,121.8992,0.0000
age_centered,"24,353.3464",1.0000,120.7420,0.0000
Residual,"206,941.5357","1,026.0000",NaN,NaN


                            OLS Regression Results                            
Dep. Variable:               strength   R-squared:                       0.279
Model:                            OLS   Adj. R-squared:                  0.277
Method:                 Least Squares   F-statistic:                     132.6
Date:                Sat, 05 Sep 2026   Prob (F-statistic):           1.36e-72
Time:                        11:08:11   Log-Likelihood:                -4192.5
No. Observations:                1030   AIC:                             8393.
Df Residuals:                    1026   BIC:                             8413.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

### 9.1 Check the homogeneous-slopes assumption

Standard ANCOVA assumes the covariate slope is common across treatment groups.

We compare:

\[
Y \sim 	ext{group}+	ext{age}
\]

with

\[
Y \sim 	ext{group}	imes	ext{age}.
\]

A meaningful interaction suggests that the age-strength slope changes by group.

In [24]:
ancova_interaction = smf.ols(
    "strength ~ C(cement_level) * age_centered",
    data=ancova_df,
).fit()

display(anova_lm(ancova_interaction, typ=2))

,sum_sq,df,F,PR(>F)
C(cement_level),"49,173.5153",2.0000,126.2701,0.0000
age_centered,"24,353.3464",1.0000,125.0714,0.0000
C(cement_level):age_centered,"7,552.7702",2.0000,19.3944,0.0000
Residual,"199,388.7655","1,024.0000",NaN,NaN


# Interlude — Latin square design

## 9.2 Controlling two nuisance dimensions

A randomized block design controls one major nuisance factor. A **Latin square** controls two.

Imagine a concrete laboratory comparing four curing protocols:

- T1
- T2
- T3
- T4

but measurements may also differ by:

- **day** — row nuisance factor
- **testing machine** — column nuisance factor

A \(4\times4\) Latin square places every treatment exactly once in every row and every column.

This section is simulated because the Kaggle file does not contain a documented Latin-square assignment.
The baseline response is anchored to the observed concrete-strength distribution.

In [25]:
treatments = np.array([
    ["T1", "T2", "T3", "T4"],
    ["T2", "T3", "T4", "T1"],
    ["T3", "T4", "T1", "T2"],
    ["T4", "T1", "T2", "T3"],
])

latin_layout = pd.DataFrame(
    treatments,
    index=[f"Day {i}" for i in range(1, 5)],
    columns=[f"Machine {j}" for j in range(1, 5)],
)
display(latin_layout)

,Machine 1,Machine 2,Machine 3,Machine 4
Day 1,T1,T2,T3,T4
Day 2,T2,T3,T4,T1
Day 3,T3,T4,T1,T2
Day 4,T4,T1,T2,T3


The additive Latin-square model is

\[
Y_{ijk} = \mu + \tau_i + \rho_j + \gamma_k + \varepsilon_{ijk},
\]

where:

- \(\tau_i\): treatment effect,
- \(\rho_j\): row/day effect,
- \(\gamma_k\): column/machine effect.

The design economizes experimental runs by balancing treatments across both nuisance dimensions.

In [26]:
latin_rows = []
overall = float(df["strength"].mean())

treatment_effect = {"T1": -4.0, "T2": 0.0, "T3": 3.0, "T4": 7.0}
day_effect = {"Day 1": -2.0, "Day 2": 1.5, "Day 3": -1.0, "Day 4": 1.5}
machine_effect = {"Machine 1": -1.5, "Machine 2": 1.0, "Machine 3": 0.5, "Machine 4": 0.0}

for i, day in enumerate(latin_layout.index):
    for j, machine in enumerate(latin_layout.columns):
        trt = latin_layout.loc[day, machine]
        y_obs = (
            overall
            + treatment_effect[trt]
            + day_effect[day]
            + machine_effect[machine]
            + rng.normal(0, 1.5)
        )
        latin_rows.append({
            "day": day,
            "machine": machine,
            "treatment": trt,
            "strength": y_obs,
        })

latin_df = pd.DataFrame(latin_rows)
display(latin_df)

,day,machine,treatment,strength
0,Day 1,Machine 1,T1,28.7750
1,Day 1,Machine 2,T2,33.2580
2,Day 1,Machine 3,T3,38.4436
3,Day 1,Machine 4,T4,42.2288
4,Day 2,Machine 1,T2,32.8914
5,Day 2,Machine 2,T3,39.3647
6,Day 2,Machine 3,T4,45.0097
7,Day 2,Machine 4,T1,32.8436
8,Day 3,Machine 1,T3,36.2928
9,Day 3,Machine 2,T4,41.5384


In [27]:
latin_model = smf.ols(
    "strength ~ C(treatment) + C(day) + C(machine)",
    data=latin_df,
).fit()

display(anova_lm(latin_model, typ=2))

,sum_sq,df,F,PR(>F)
C(treatment),238.1855,3.0000,54.7119,0.0001
C(day),25.7963,3.0000,5.9255,0.0316
C(machine),24.9989,3.0000,5.7423,0.0338
Residual,8.7069,6.0000,NaN,NaN


### Interpretation

The treatment comparison is made **after accounting for both day and machine variation**.

A Latin square depends on restrictive assumptions, especially an additive structure with no estimable
treatment-by-row or treatment-by-column interactions in the basic unreplicated design.

That is why experimental design is a trade-off between **efficiency** and **what interactions remain estimable**.

# Part III — Multi-factor experiments and interactions

## 10. Two-factor factorial-style analysis on the observed data

We now create low/high analysis strata for:

- **A = cement**
- **B = water**

To make the two levels more distinct, use lower and upper 35% regions and discard the middle 30%.
This mimics a two-level factor analysis, but it is still observational.

The two-factor model is

\[
Y=\beta_0+\beta_A A+\beta_B B+\beta_{AB}AB+\varepsilon.
\]

The interaction term asks:

> Does the effect associated with cement depend on the water level?

In [28]:
def extreme_binary_level(
    s: pd.Series,
    low_label: str = "Low",
    high_label: str = "High",
    q_low: float = 0.35,
    q_high: float = 0.65,
) -> pd.Series:
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)

    out = pd.Series(pd.NA, index=s.index, dtype="object")
    out.loc[s <= lo] = low_label
    out.loc[s >= hi] = high_label
    return out


two_factor_df = df.copy()
two_factor_df["A_cement"] = extreme_binary_level(two_factor_df["cement"])
two_factor_df["B_water"] = extreme_binary_level(two_factor_df["water"])
two_factor_df = two_factor_df.dropna(subset=["A_cement", "B_water"]).copy()

display(pd.crosstab(two_factor_df["A_cement"], two_factor_df["B_water"]))

B_water,High,Low
A_cement,,
High,125,154
Low,125,135


In [29]:
two_factor_model = smf.ols(
    "strength ~ C(A_cement) * C(B_water)",
    data=two_factor_df,
).fit()

display(anova_lm(two_factor_model, typ=2))

,sum_sq,df,F,PR(>F)
C(A_cement),"40,542.9559",1.0000,218.6963,0.0000
C(B_water),"24,613.2189",1.0000,132.7683,0.0000
C(A_cement):C(B_water),"3,163.1130",1.0000,17.0624,0.0000
Residual,"99,180.8203",535.0000,NaN,NaN


### 10.1 Interaction plot

Non-parallel lines are a visual warning that the effect of one factor depends on the level of another.

In [30]:
interaction_means = (
    two_factor_df.groupby(["A_cement", "B_water"], observed=True)["strength"]
    .mean()
    .reset_index()
)

p = figure(
    x_range=["Low", "High"],
    width=760,
    height=420,
    title="Interaction plot: cement × water",
    x_axis_label="Cement level",
    y_axis_label="Mean observed strength (MPa)",
    tools="pan,wheel_zoom,reset,save",
)

for water_level, g in interaction_means.groupby("B_water"):
    g = g.set_index("A_cement").reindex(["Low", "High"]).reset_index()
    p.line(
        x=g["A_cement"],
        y=g["strength"],
        line_width=3,
        legend_label=f"Water={water_level}",
    )
    p.scatter(
        x=g["A_cement"],
        y=g["strength"],
        size=9,
        legend_label=f"Water={water_level}",
    )

p.legend.location = "top_left"
show(p)

## 11. Why interaction changes interpretation

Suppose the interaction is important.

Then saying simply:

> "High cement is associated with +X MPa"

can be misleading.

The correct statement becomes conditional:

> "The cement effect differs depending on the water setting."

This is a core reason factorial experiments are superior to one-factor-at-a-time experimentation.

# Part IV — Build a genuine \(2^k\) design anchored to the Kaggle factor ranges

The next sections separate **experimental design** from the historical dataset.

We will:

1. fit a flexible response-surface surrogate to the Kaggle data,
2. choose realistic low/high factor values from the observed 25th/75th percentiles,
3. generate a balanced \(2^3\) design,
4. randomize run order,
5. replicate runs,
6. use the surrogate only as a stand-in for a physical laboratory response.

This gives us a clean way to study factorial-design mathematics without pretending that the original
data was randomized.

## 12. Fit a response-surface surrogate

The surrogate is **not the scientific conclusion**. It is simply our synthetic laboratory.

A real experiment would physically produce concrete mixtures at each assigned factor combination.

In [31]:
feature_cols = [
    "cement",
    "slag",
    "fly_ash",
    "water",
    "superplasticizer",
    "coarse_aggregate",
    "fine_aggregate",
    "age",
]

X = df[feature_cols]
y = df["strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

oracle = RandomForestRegressor(
    n_estimators=400,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
oracle.fit(X_train, y_train)

pred = oracle.predict(X_test)
print(f"Surrogate holdout R^2: {r2_score(y_test, pred):.4f}")

Surrogate holdout R^2: 0.8745


## 13. Construct a balanced \(2^3\) factorial

Factors:

- \(A\): cement
- \(B\): water
- \(C\): slag

Each factor has coded levels:

\[
-1 = \text{low},\qquad +1 = \text{high}.
\]

A full design requires

\[
2^3=8
\]

unique treatment combinations.

In [32]:
FACTOR_MAP = {
    "A": "cement",
    "B": "water",
    "C": "slag",
}

levels = {
    col: {
        -1: float(df[col].quantile(0.25)),
        +1: float(df[col].quantile(0.75)),
    }
    for col in FACTOR_MAP.values()
}

display(pd.DataFrame(levels).T.rename(columns={-1: "low (-1)", 1: "high (+1)"}))

,low (-1),high (+1)
cement,192.3750,350.0000
water,164.9000,192.0000
slag,0.0000,142.9500


In [33]:
design_2k = pd.DataFrame(
    list(product([-1, 1], repeat=3)),
    columns=["A", "B", "C"],
)

design_2k["AB"] = design_2k["A"] * design_2k["B"]
design_2k["AC"] = design_2k["A"] * design_2k["C"]
design_2k["BC"] = design_2k["B"] * design_2k["C"]
design_2k["ABC"] = design_2k["A"] * design_2k["B"] * design_2k["C"]

display(design_2k)

,A,B,C,AB,AC,BC,ABC
0,-1,-1,-1,1,1,1,-1
1,-1,-1,1,1,-1,-1,1
2,-1,1,-1,-1,1,-1,1
3,-1,1,1,-1,-1,1,-1
4,1,-1,-1,-1,-1,1,1
5,1,-1,1,-1,1,-1,-1
6,1,1,-1,1,-1,-1,-1
7,1,1,1,1,1,1,1


### 13.1 Orthogonality check

For a balanced two-level full factorial design, effect columns are orthogonal.

If \(X\) contains the coded effect columns, then off-diagonal entries of

\[
X^	op X
\]

are zero.

In [34]:
effect_cols = ["A", "B", "C", "AB", "AC", "BC", "ABC"]
xtx = design_2k[effect_cols].T @ design_2k[effect_cols]
display(xtx)

,A,B,C,AB,AC,BC,ABC
A,8,0,0,0,0,0,0
B,0,8,0,0,0,0,0
C,0,0,8,0,0,0,0
AB,0,0,0,8,0,0,0
AC,0,0,0,0,8,0,0
BC,0,0,0,0,0,8,0
ABC,0,0,0,0,0,0,8


### 13.2 Map coded settings to physical values

Variables not manipulated in this experiment are held at their medians.

This is exactly the sort of decision that must be made **before** a physical experiment.

In [35]:
baseline = df[feature_cols].median().to_dict()

def coded_to_physical(row: pd.Series) -> dict:
    x = baseline.copy()
    for code_name, physical_name in FACTOR_MAP.items():
        x[physical_name] = levels[physical_name][int(row[code_name])]
    return x


physical_design = pd.DataFrame(
    [coded_to_physical(row) for _, row in design_2k.iterrows()]
)

full_design = pd.concat(
    [design_2k.reset_index(drop=True), physical_design.reset_index(drop=True)],
    axis=1,
)

display(full_design)

,A,B,C,AB,AC,BC,ABC,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age
0,-1,-1,-1,1,1,1,-1,192.3750,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000
1,-1,-1,1,1,-1,-1,1,192.3750,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000
2,-1,1,-1,-1,1,-1,1,192.3750,0.0000,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000
3,-1,1,1,-1,-1,1,-1,192.3750,142.9500,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000
4,1,-1,-1,-1,-1,1,1,350.0000,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000
5,1,-1,1,-1,1,-1,-1,350.0000,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000
6,1,1,-1,1,-1,-1,-1,350.0000,0.0000,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000
7,1,1,1,1,1,1,1,350.0000,142.9500,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000


## 14. Replication and randomization

We take each of the 8 treatment combinations and replicate it 4 times:

\[
N=8	imes4=32.
\]

Then we randomize run order.

Replication gives us an estimate of experimental error; randomization protects against systematic
run-order effects.

In [36]:
N_REPLICATES = 4

runs = pd.concat(
    [full_design.assign(replicate=r + 1) for r in range(N_REPLICATES)],
    ignore_index=True,
)

runs = runs.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
runs["run_order"] = np.arange(1, len(runs) + 1)

display(runs.head(12))

,A,B,C,AB,AC,BC,ABC,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,replicate,run_order
0,1,-1,1,-1,1,-1,-1,350.0000,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,4,1
1,1,1,1,1,1,1,1,350.0000,142.9500,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000,2,2
2,-1,-1,-1,1,1,1,-1,192.3750,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,4,3
3,-1,-1,1,1,-1,-1,1,192.3750,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,3,4
4,-1,-1,-1,1,1,1,-1,192.3750,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,2,5
5,-1,-1,1,1,-1,-1,1,192.3750,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,2,6
6,1,1,-1,1,-1,-1,-1,350.0000,0.0000,0.0000,192.0000,6.4000,968.0000,779.5000,28.0000,4,7
7,-1,-1,1,1,-1,-1,1,192.3750,142.9500,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,4,8
8,1,-1,-1,-1,-1,1,1,350.0000,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,2,9
9,-1,-1,-1,1,1,1,-1,192.3750,0.0000,0.0000,164.9000,6.4000,968.0000,779.5000,28.0000,1,10


### 14.1 Generate synthetic laboratory outcomes

The random-forest surrogate predicts the expected response at each setting.
We then add independent measurement/process noise.

The noise is what replication lets us estimate.

In [37]:
expected_strength = oracle.predict(runs[feature_cols])

# A modest synthetic process-noise SD, chosen for teaching.
PROCESS_SD = 2.5
runs["strength"] = expected_strength + rng.normal(0, PROCESS_SD, size=len(runs))

display(runs[["run_order", "replicate", "A", "B", "C", "cement", "water", "slag", "strength"]].head(12))

,run_order,replicate,A,B,C,cement,water,slag,strength
0,1,4,1,-1,1,350.0000,164.9000,142.9500,59.5790
1,2,2,1,1,1,350.0000,192.0000,142.9500,41.9141
2,3,4,-1,-1,-1,192.3750,164.9000,0.0000,33.6999
3,4,3,-1,-1,1,192.3750,164.9000,142.9500,41.3554
4,5,2,-1,-1,-1,192.3750,164.9000,0.0000,31.0416
5,6,2,-1,-1,1,192.3750,164.9000,142.9500,39.7778
6,7,4,1,1,-1,350.0000,192.0000,0.0000,39.3063
7,8,4,-1,-1,1,192.3750,164.9000,142.9500,41.0938
8,9,2,1,-1,-1,350.0000,164.9000,0.0000,53.0140
9,10,1,-1,-1,-1,192.3750,164.9000,0.0000,30.6234


In [38]:
p = figure(
    width=800,
    height=400,
    title="Randomized run order",
    x_axis_label="Run order",
    y_axis_label="Observed synthetic strength (MPa)",
    tools="pan,wheel_zoom,reset,save,hover",
    tooltips=[
        ("run", "@run_order"),
        ("A", "@A"),
        ("B", "@B"),
        ("C", "@C"),
        ("strength", "@strength{0.00}"),
    ],
)
p.scatter(
    "run_order",
    "strength",
    source=ColumnDataSource(runs),
    size=7,
    alpha=0.7,
)
show(p)

## 15. Estimate main effects and interactions

With \(-1/+1\) coding:

\[
\text{Effect}(A)=\bar Y_{A=+1}-\bar Y_{A=-1}.
\]

Equivalent regression:

\[
Y=\beta_0+\beta_A A+\beta_B B+\cdots
\]

With this coding,

\[
\text{Effect}(A)=2\beta_A.
\]

In [39]:
factorial_model = smf.ols(
    "strength ~ A * B * C",
    data=runs,
).fit()

print(factorial_model.summary())
display(anova_lm(factorial_model, typ=2))

                            OLS Regression Results                            
Dep. Variable:               strength   R-squared:                       0.978
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                     155.3
Date:                Sat, 05 Sep 2026   Prob (F-statistic):           2.07e-18
Time:                        11:08:13   Log-Likelihood:                -58.762
No. Observations:                  32   AIC:                             133.5
Df Residuals:                      24   BIC:                             145.3
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     41.1689      0.310    132.865      0.0

,sum_sq,df,F,PR(>F)
A,"1,741.9104",1.0000,566.9681,0.0000
B,954.5032,1.0000,310.6778,0.0000
A:B,283.4782,1.0000,92.2683,0.0000
C,342.8720,1.0000,111.6001,0.0000
A:C,4.9962,1.0000,1.6262,0.2144
B:C,0.1629,1.0000,0.0530,0.8198
A:B:C,11.1668,1.0000,3.6346,0.0686
Residual,73.7358,24.0000,NaN,NaN


In [40]:
def factorial_effects(model, effects: Sequence[str]) -> pd.DataFrame:
    rows = []
    for effect in effects:
        coef_name = effect.replace(":", ":")
        beta = model.params.get(coef_name, np.nan)
        rows.append({
            "effect": effect,
            "beta_coefficient": beta,
            "DOE_effect_2beta": 2 * beta,
        })
    return pd.DataFrame(rows)


effects_df = factorial_effects(
    factorial_model,
    ["A", "B", "C", "A:B", "A:C", "B:C", "A:B:C"],
)
display(effects_df)

,effect,beta_coefficient,DOE_effect_2beta
0,A,7.3780,14.7560
1,B,-5.4615,-10.9230
2,C,3.2733,6.5467
3,A:B,-2.9764,-5.9527
4,A:C,-0.3951,-0.7903
5,B:C,-0.0714,-0.1427
6,A:B:C,0.5907,1.1815


In [41]:
# Rank effects by absolute magnitude.
plot_effects = effects_df.copy()
plot_effects["abs_effect"] = plot_effects["DOE_effect_2beta"].abs()
plot_effects = plot_effects.sort_values("abs_effect")

p = figure(
    y_range=plot_effects["effect"].tolist(),
    width=800,
    height=420,
    title="Estimated factorial effects",
    x_axis_label="Effect on strength (MPa)",
    y_axis_label="Factor / interaction",
    tools="pan,wheel_zoom,reset,save,hover",
    tooltips=[("effect", "@effect"), ("estimate", "@DOE_effect_2beta{0.000}")],
)
p.hbar(
    y="effect",
    right="DOE_effect_2beta",
    height=0.6,
    source=ColumnDataSource(plot_effects),
    alpha=0.7,
)
p.add_layout(Span(location=0, dimension="height", line_dash="dashed"))
show(p)

## 16. Main-effects plots

For each factor, compare the average response at coded level -1 and +1.

In [42]:
main_effect_rows = []
for factor in ["A", "B", "C"]:
    means = runs.groupby(factor)["strength"].mean()
    main_effect_rows.append({
        "factor": factor,
        "low_mean": means.loc[-1],
        "high_mean": means.loc[1],
        "effect": means.loc[1] - means.loc[-1],
    })

main_effects = pd.DataFrame(main_effect_rows)
display(main_effects)

,factor,low_mean,high_mean,effect
0,A,33.7909,48.5469,14.7560
1,B,46.6304,35.7074,-10.9230
2,C,37.8956,44.4423,6.5467


In [43]:
plots = []
for factor in ["A", "B", "C"]:
    g = (
        runs.groupby(factor)["strength"]
        .mean()
        .reindex([-1, 1])
        .reset_index()
    )

    p = figure(
        width=350,
        height=330,
        title=f"Main effect: {factor}",
        x_axis_label=f"{factor} coded level",
        y_axis_label="Mean strength",
        tools="reset,save",
    )
    p.line(g[factor], g["strength"], line_width=3)
    p.scatter(g[factor], g["strength"], size=9)
    plots.append(p)

show(gridplot([plots]))

# Interlude — Random effects and variance components

## 16.1 Fixed effects vs random effects

The factorial coefficients \(A,B,C\) describe specific settings deliberately chosen for study, so they
are naturally **fixed effects**.

Now suppose the 4 replicates were performed in 4 production batches randomly sampled from a much larger
population of possible batches. We may model

\[
b_j \sim N(0,\sigma_b^2)
\]

and

\[
Y_{ij} = X_{ij}\beta + b_j + \varepsilon_{ij}.
\]

The goal is no longer to estimate a separate scientific coefficient for every possible batch. Instead,
we estimate the **between-batch variance component** \(\sigma_b^2\).

In [44]:
mixed_runs = runs.copy()
mixed_runs["batch"] = "Batch " + mixed_runs["replicate"].astype(str)

# Add a synthetic batch-to-batch deviation.
batch_levels = sorted(mixed_runs["batch"].unique())
batch_offsets = {
    b: offset
    for b, offset in zip(
        batch_levels,
        rng.normal(0, 2.0, size=len(batch_levels)),
    )
}

mixed_runs["strength_mixed"] = (
    mixed_runs["strength"]
    + mixed_runs["batch"].map(batch_offsets).astype(float)
)

display(
    mixed_runs[
        ["batch", "A", "B", "C", "strength", "strength_mixed"]
    ].head(12)
)

,batch,A,B,C,strength,strength_mixed
0,Batch 4,1,-1,1,59.5790,60.8416
1,Batch 2,1,1,1,41.9141,42.0493
2,Batch 4,-1,-1,-1,33.6999,34.9624
3,Batch 3,-1,-1,1,41.3554,41.9336
4,Batch 2,-1,-1,-1,31.0416,31.1767
5,Batch 2,-1,-1,1,39.7778,39.9130
6,Batch 4,1,1,-1,39.3063,40.5689
7,Batch 4,-1,-1,1,41.0938,42.3564
8,Batch 2,1,-1,-1,53.0140,53.1492
9,Batch 1,-1,-1,-1,30.6234,31.9812


In [45]:
mixed_model = smf.mixedlm(
    "strength_mixed ~ A * B * C",
    data=mixed_runs,
    groups=mixed_runs["batch"],
)

mixed_result = mixed_model.fit(reml=True)
print(mixed_result.summary())

batch_variance = float(mixed_result.cov_re.iloc[0, 0])
residual_variance = float(mixed_result.scale)
icc = batch_variance / (batch_variance + residual_variance)

print(f"\nEstimated batch variance:    {batch_variance:.4f}")
print(f"Estimated residual variance: {residual_variance:.4f}")
print(f"Approximate ICC:              {icc:.4f}")

           Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: strength_mixed
No. Observations: 32      Method:             REML          
No. Groups:       4       Scale:              3.1575        
Min. group size:  8       Log-Likelihood:     -63.0454      
Max. group size:  8       Converged:          No            
Mean group size:  8.0                                       
-------------------------------------------------------------
             Coef.   Std.Err.     z     P>|z|  [0.025  0.975]
-------------------------------------------------------------
Intercept    42.002     0.489   85.815  0.000  41.043  42.962
A             7.378     0.314   23.488  0.000   6.762   7.994
B            -5.462     0.314  -17.387  0.000  -6.077  -4.846
A:B          -2.976     0.314   -9.475  0.000  -3.592  -2.361
C             3.273     0.314   10.421  0.000   2.658   3.889
A:C          -0.395     0.314   -1.258  0.208  -1.011   0.221
B:C          -0.071     0.3

### Why this matters

A fixed-effect question is:

> How does the response differ between these explicitly chosen factor levels?

A random-effect question is:

> How much variability is contributed by a population of batches, machines, laboratories, operators, etc.?

This leads naturally to mixed models and hierarchical models.

The intraclass correlation

\[
ICC=\frac{\sigma_b^2}{\sigma_b^2+\sigma^2}
\]

summarizes how much total residual variability is attributable to between-batch differences in this
simple random-intercept setting.

# Part V — Blocking and confounding in factorial designs

## 17. What if the experiment must be run in two batches?

Suppose only 4 of the 8 unique treatment combinations can be run per day.

We can divide the \(2^3\) design into two blocks.

A common strategy is to deliberately confound a high-order interaction with block.

For illustration, define the block using

\[
ABC.
\]

Then the \(ABC\) interaction is not separately estimable from the block effect.

This is acceptable only if sacrificing \(ABC\) is scientifically reasonable.

In [46]:
blocked_unique = design_2k.copy()
blocked_unique["block"] = np.where(blocked_unique["ABC"] == -1, "Block 1", "Block 2")
display(blocked_unique)

,A,B,C,AB,AC,BC,ABC,block
0,-1,-1,-1,1,1,1,-1,Block 1
1,-1,-1,1,1,-1,-1,1,Block 2
2,-1,1,-1,-1,1,-1,1,Block 2
3,-1,1,1,-1,-1,1,-1,Block 1
4,1,-1,-1,-1,-1,1,1,Block 2
5,1,-1,1,-1,1,-1,-1,Block 1
6,1,1,-1,1,-1,-1,-1,Block 1
7,1,1,1,1,1,1,1,Block 2


In [47]:
display(pd.crosstab(blocked_unique["block"], [blocked_unique["A"], blocked_unique["B"], blocked_unique["C"]]))

A       -1           1         
B       -1     1    -1     1   
C       -1  1 -1  1 -1  1 -1  1
block                          
Block 1  1  0  0  1  0  1  1  0
Block 2  0  1  1  0  1  0  0  1

### Defining relation

If block is constructed from \(ABC\), then the design deliberately aliases the block contrast with \(ABC\).

The core engineering idea is:

> spend your degrees of freedom on scientifically important effects and sacrifice effects believed to be negligible.

This same logic motivates fractional factorial designs.

# Part VI — Fractional factorial design

## 18. A \(2^{4-1}\) half-fraction

Suppose we want four factors:

- \(A\): cement
- \(B\): water
- \(C\): slag
- \(D\): superplasticizer

A full \(2^4\) design needs 16 unique runs.

A half-fraction uses only

\[
2^{4-1}=8
\]

runs.

Choose the generator

\[
D=ABC.
\]

Multiplying both sides by \(D\):

\[
I=ABCD.
\]

This defining relation determines the alias structure.

In [48]:
frac = pd.DataFrame(
    list(product([-1, 1], repeat=3)),
    columns=["A", "B", "C"],
)
frac["D"] = frac["A"] * frac["B"] * frac["C"]

display(frac)
print("Unique runs:", len(frac))

,A,B,C,D
0,-1,-1,-1,-1
1,-1,-1,1,1
2,-1,1,-1,1
3,-1,1,1,-1
4,1,-1,-1,1
5,1,-1,1,-1
6,1,1,-1,-1
7,1,1,1,1


Unique runs: 8


## 19. Derive the alias structure

With

\[
I=ABCD,
\]

multiply an effect by \(ABCD\):

\[
A \equiv BCD
\]

\[
B \equiv ACD
\]

\[
C \equiv ABD
\]

\[
D \equiv ABC
\]

and for two-factor interactions:

\[
AB \equiv CD,\qquad AC\equiv BD,\qquad AD\equiv BC.
\]

This is a **Resolution IV** design:

- main effects are aliased with 3-factor interactions,
- 2-factor interactions are aliased with other 2-factor interactions.

In [49]:
alias_table = pd.DataFrame({
    "estimand": ["A", "B", "C", "D", "AB", "AC", "AD"],
    "aliased_with": ["BCD", "ACD", "ABD", "ABC", "CD", "BD", "BC"],
})
display(alias_table)

,estimand,aliased_with
0,A,BCD
1,B,ACD
2,C,ABD
3,D,ABC
4,AB,CD
5,AC,BD
6,AD,BC


## 20. Map the fractional design to realistic concrete settings

Again, low/high settings come from empirical quartiles in the Kaggle data.

In [50]:
FACTOR_MAP_4 = {
    "A": "cement",
    "B": "water",
    "C": "slag",
    "D": "superplasticizer",
}

levels_4 = {
    col: {
        -1: float(df[col].quantile(0.25)),
        +1: float(df[col].quantile(0.75)),
    }
    for col in FACTOR_MAP_4.values()
}

frac_physical = []
for _, row in frac.iterrows():
    x = baseline.copy()
    for code_name, physical_name in FACTOR_MAP_4.items():
        x[physical_name] = levels_4[physical_name][int(row[code_name])]
    frac_physical.append(x)

frac_design = pd.concat(
    [frac.reset_index(drop=True), pd.DataFrame(frac_physical)],
    axis=1,
)

frac_design["expected_strength"] = oracle.predict(frac_design[feature_cols])
display(frac_design)

,A,B,C,D,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,expected_strength
0,-1,-1,-1,-1,192.3750,0.0000,0.0000,164.9000,0.0000,968.0000,779.5000,28.0000,28.4631
1,-1,-1,1,1,192.3750,142.9500,0.0000,164.9000,10.2000,968.0000,779.5000,28.0000,41.9168
2,-1,1,-1,1,192.3750,0.0000,0.0000,192.0000,10.2000,968.0000,779.5000,28.0000,28.0995
3,-1,1,1,-1,192.3750,142.9500,0.0000,192.0000,0.0000,968.0000,779.5000,28.0000,30.1312
4,1,-1,-1,1,350.0000,0.0000,0.0000,164.9000,10.2000,968.0000,779.5000,28.0000,53.9695
5,1,-1,1,-1,350.0000,142.9500,0.0000,164.9000,0.0000,968.0000,779.5000,28.0000,58.3716
6,1,1,-1,-1,350.0000,0.0000,0.0000,192.0000,0.0000,968.0000,779.5000,28.0000,32.8777
7,1,1,1,1,350.0000,142.9500,0.0000,192.0000,10.2000,968.0000,779.5000,28.0000,46.3462


### 20.1 Why this is efficient

Full \(2^4\):

\[
16 	ext{ unique combinations}
\]

Half-fraction:

\[
8 	ext{ unique combinations}
\]

Run-count reduction:

\[
50\%.
\]

But the information loss appears as **aliasing**.

DOE is therefore an explicit information/cost trade-off.

# Part VII — Power and sample-size thinking

## 21. Replication is not arbitrary

Suppose a future one-way experiment has 3 treatment groups.

For standardized ANOVA effect size \(f\), significance level \(\alpha=0.05\), and desired power \(0.8\),
we can estimate how many total observations are required.

Cohen's \(f\) is related to between-group variation relative to within-group noise.

In [51]:
power_solver = FTestAnovaPower()

effect_sizes = [0.10, 0.25, 0.40]
power_rows = []

for f in effect_sizes:
    n_total = power_solver.solve_power(
        effect_size=f,
        k_groups=3,
        alpha=0.05,
        power=0.80,
    )
    power_rows.append({
        "Cohen_f": f,
        "total_n_required": int(np.ceil(n_total)),
        "approx_per_group": int(np.ceil(n_total / 3)),
    })

power_table = pd.DataFrame(power_rows)
display(power_table)

,Cohen_f,total_n_required,approx_per_group
0,0.1000,967,323
1,0.2500,158,53
2,0.4000,64,22


In [52]:
n_grid = np.arange(15, 500)

p = figure(
    width=800,
    height=430,
    title="ANOVA power as total sample size increases",
    x_axis_label="Total sample size",
    y_axis_label="Power",
    tools="pan,wheel_zoom,reset,save",
)

for f in effect_sizes:
    powers = [
        power_solver.power(
            effect_size=f,
            nobs=n,
            alpha=0.05,
            k_groups=3,
        )
        for n in n_grid
    ]
    p.line(n_grid, powers, line_width=2, legend_label=f"f={f}")

p.add_layout(Span(location=0.80, dimension="width", line_dash="dashed"))
p.legend.location = "bottom_right"
show(p)

### Practical interpretation

Small effects require much more replication than large effects.

Therefore a good experiment starts with:

- smallest practically meaningful effect,
- expected process variance,
- desired power,
- cost per run,
- feasible number of experimental units.

**"Collect as much data as possible" is not an experimental-design strategy.**

# Part VIII — Causal inference: randomized assignment vs confounding

## 22. Why randomization matters

The raw Kaggle dataset cannot by itself identify all causal effects because treatment assignment is not
documented as randomized.

To demonstrate the causal idea cleanly, we create a hypothetical curing intervention:

- \(T=0\): standard protocol
- \(T=1\): enhanced protocol

Assume the true causal effect is \(+5\) MPa.

We compare:

1. randomized treatment assignment,
2. confounded assignment where stronger-baseline mixtures are more likely to receive treatment.

In [53]:
causal_base = df.sample(600, random_state=RANDOM_STATE).copy().reset_index(drop=True)
TRUE_EFFECT = 5.0

# Potential outcome baseline with some measurement noise.
causal_base["Y0"] = causal_base["strength"] + rng.normal(0, 1.0, size=len(causal_base))
causal_base["Y1"] = causal_base["Y0"] + TRUE_EFFECT

# 1) Randomized assignment.
causal_base["T_random"] = rng.binomial(1, 0.5, size=len(causal_base))
causal_base["Y_random"] = np.where(
    causal_base["T_random"] == 1,
    causal_base["Y1"],
    causal_base["Y0"],
)

randomized_estimate = (
    causal_base.loc[causal_base["T_random"] == 1, "Y_random"].mean()
    - causal_base.loc[causal_base["T_random"] == 0, "Y_random"].mean()
)

# 2) Confounded assignment:
# high cement / older concrete is more likely to receive treatment.
score = (
    0.015 * (causal_base["cement"] - causal_base["cement"].mean())
    + 0.01 * (causal_base["age"] - causal_base["age"].mean())
)
prob = 1 / (1 + np.exp(-score))
causal_base["T_confounded"] = rng.binomial(1, prob.clip(0.05, 0.95))
causal_base["Y_confounded"] = np.where(
    causal_base["T_confounded"] == 1,
    causal_base["Y1"],
    causal_base["Y0"],
)

confounded_estimate = (
    causal_base.loc[causal_base["T_confounded"] == 1, "Y_confounded"].mean()
    - causal_base.loc[causal_base["T_confounded"] == 0, "Y_confounded"].mean()
)

print(f"True causal effect:                 {TRUE_EFFECT:.3f} MPa")
print(f"Randomized difference-in-means:    {randomized_estimate:.3f} MPa")
print(f"Confounded difference-in-means:    {confounded_estimate:.3f} MPa")

True causal effect:                 5.000 MPa
Randomized difference-in-means:    2.827 MPa
Confounded difference-in-means:    14.525 MPa


### 22.1 Adjustment can help observational comparisons — but requires assumptions

For the confounded sample, fit:

\[
Y \sim T + 	ext{cement}+	ext{age}.
\]

If the confounders are measured and the model is correctly specified, adjustment may reduce bias.

But unlike randomization, observational adjustment relies on stronger assumptions:

- no important unmeasured confounding,
- correct model structure,
- overlap/positivity,
- consistent treatment definition.

In [54]:
confounded_adjusted = smf.ols(
    "Y_confounded ~ T_confounded + cement + age",
    data=causal_base,
).fit()

print(confounded_adjusted.summary())
print(
    "\nAdjusted treatment coefficient:",
    f"{confounded_adjusted.params['T_confounded']:.3f} MPa"
)

                            OLS Regression Results                            
Dep. Variable:           Y_confounded   R-squared:                       0.391
Model:                            OLS   Adj. R-squared:                  0.388
Method:                 Least Squares   F-statistic:                     127.6
Date:                Sat, 05 Sep 2026   Prob (F-statistic):           7.82e-64
Time:                        11:08:14   Log-Likelihood:                -2433.8
No. Observations:                 600   AIC:                             4876.
Df Residuals:                     596   BIC:                             4893.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        9.9653      1.721      5.791   

# Part IX — A compact DOE decision framework

## 23. From engineering question to experimental design

Use this sequence before writing analysis code.

### Step 1 — Define the response

Examples:

- concrete compressive strength,
- Spark pipeline runtime,
- model latency,
- conversion rate,
- RAG answer quality.

### Step 2 — Identify controllable factors

Examples in this case:

- cement,
- water,
- slag,
- superplasticizer.

### Step 3 — Identify nuisance variables

Examples:

- curing age,
- production batch,
- machine,
- operator,
- day,
- data center.

### Step 4 — Pick meaningful levels

Avoid selecting levels only because they are mathematically convenient.

Use domain constraints and feasibility.

### Step 5 — Choose a design

- 1 factor → completely randomized one-way experiment
- nuisance block → randomized block design
- 2–5 two-level factors → full factorial may be feasible
- many screening factors → fractional factorial
- quantitative optimization → response-surface methodology (next topic beyond this notebook's core scope)

### Step 6 — Determine replication / power

Specify the minimum practically important effect.

### Step 7 — Randomize run order

Protect against drift and hidden time trends.

### Step 8 — Collect data exactly according to protocol

Do not quietly modify treatment assignment mid-experiment.

### Step 9 — Analyse using the design structure

Use treatment, block, interaction and covariance terms that correspond to the design.

### Step 10 — Diagnose assumptions

Inspect residuals and sensitivity.

### Step 11 — Report statistical + practical significance

A tiny p-value is not automatically an engineering win.

### Step 12 — Separate association from causal interpretation

Causal language must be supported by the assignment mechanism and assumptions.

# Part X — Optional exercises

Try these before reading the suggested solution ideas.

## Exercise 1 — Different cement strata

Change the one-way ANOVA from tertiles to quartiles.

Questions:

1. How does the ANOVA table change?
2. Does \(\eta^2\) increase or decrease?
3. Are residual assumptions improved?
4. Why can discretizing a continuous variable lose information?

---

## Exercise 2 — Different blocking variable

Use a coarse `water` stratum as a block-like variable and compare residual variance with the unblocked model.

What makes a variable a **good block**?

---

## Exercise 3 — Three-factor observational interaction

Construct low/high strata for:

- cement,
- water,
- superplasticizer

and fit:

```python
strength ~ A * B * C
```

Inspect which interactions are plausible.

Why should the result still not be called a randomized causal experiment?

---

## Exercise 4 — Fractional factorial generator

For a \(2^{5-2}\) design, experiment with generators such as:

\[
D=AB,\qquad E=AC.
\]

Derive the defining relation and investigate the alias chains.

---

## Exercise 5 — Power

Recompute required sample size for:

- power = 0.90
- \(lpha=0.01\)
- 4 groups

Explain why stronger evidence requirements increase the sample-size requirement.

# Suggested solution ideas

### Exercise 1
Using more strata can capture nonlinear structure but reduces observations per group and still throws away
within-bin information. ANCOVA/regression with continuous cement often uses information more efficiently.

### Exercise 2
A good block is strongly related to the response but is **not itself the treatment of primary interest**.
Blocking should reduce unexplained variance.

### Exercise 3
The model can estimate association/interactions in observed regions, but factor assignments were not generated
by your randomization protocol. Confounding and selection may remain.

### Exercise 4
Multiply generators together and use the resulting defining words to obtain alias chains.

### Exercise 5
Higher power and smaller alpha both demand more information, so required \(N\) increases.

# Final conceptual summary

The notebook illustrated the ST3232 progression:

\[
\text{Question}
\rightarrow
\text{Design}
\rightarrow
\text{Randomize}
\rightarrow
\text{Replicate}
\rightarrow
\text{Collect}
\rightarrow
\text{ANOVA / regression}
\rightarrow
\text{Interactions}
\rightarrow
\text{Diagnostics}
\rightarrow
\text{Causal interpretation}.
\]

The most important lesson is not the syntax of `anova_lm`.

It is this:

> **The information you can extract later is constrained by the experiment you designed earlier.**

A factorial design is powerful because it deliberately creates information about main effects and interactions.
A fractional factorial design saves runs by deliberately accepting aliasing.
Blocking saves precision by removing nuisance variation.
Randomization is what gives the strongest protection against confounding.

That is the core statistical-engineering mindset behind Design and Analysis of Experiments.

## References

1. I-Cheng Yeh, *Modeling of strength of high-performance concrete using artificial neural networks*,
   Cement and Concrete Research 28(12), 1998.
2. Kaggle dataset mirror: https://www.kaggle.com/datasets/vivekgediya/concrete-data
3. UCI Concrete Compressive Strength dataset.
4. Douglas C. Montgomery, *Design and Analysis of Experiments*.
5. `statsmodels` documentation for ANOVA, multiple comparisons and power analysis.

### Note on reproducibility

Random operations use `RANDOM_STATE = 42`. The synthetic experimental sections will therefore be
reproducible when package versions are comparable.